# Imputação do indicador faltante

Este notebook completa a matriz de decisão. Ele parte de `resultados/municipios_selecionados.csv`, gerado pelo notebook `01_filtragem_municipios.ipynb`, em que 76 dos 581 municípios têm um indicador faltante.

## Escolha da técnica

O valor faltante é estimado pelos municípios **de mesmo porte** cujo desempenho nos **demais indicadores** é mais parecido, considerando todos os municípios do Nordeste. Três famílias de técnicas foram consideradas:

- **Mediana ou média do porte:** ignora os outros indicadores e serve de referência mínima.
- **Regressão linear:** usa os outros indicadores como preditores. Supõe relação linear e pode prever fora de 0–100. Aqui as correlações entre indicadores são fracas (no máximo 0,34) e as distribuições são concentradas em 0 e 100 (IES2003, IRS3002), o que não favorece um modelo linear.
- **K vizinhos mais próximos (KNN) dentro do porte:** procura os *k* municípios de mesmo porte mais parecidos nos indicadores conhecidos e usa a média deles, ponderada pela proximidade. É a tradução direta da ideia de "municípios de mesmo porte com desempenho semelhante". Não supõe forma funcional, sempre produz valores dentro de 0–100 e permite mostrar quais municípios serviram de referência.

Uma floresta aleatória (*random forest*) entra na comparação como referência de um modelo não linear mais flexível.

A escolha é feita por **validação cruzada** (10 partes, 5 repetições). Nos municípios em que o indicador é conhecido, ele é escondido, estimado por cada técnica e comparado ao valor real.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RepeatedKFold
from sklearn.neighbors import NearestNeighbors

# O notebook fica em "jupyter notebooks/"; a raiz do repositório é a pasta que contém "dados/"
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados").is_dir())
ENTRADA = RAIZ / "resultados" / "municipios_selecionados.csv"
SAIDA = RAIZ / "resultados" / "matriz_decisao.csv"

INDICADORES = ["IAG0001", "IES0001", "IES2003", "IRS0001", "IRS3002"]
VALORES_K = [5, 10, 15, 20, 30]
SEMENTE = 0

mun = pd.read_csv(ENTRADA, dtype={"cod_IBGE": str}).set_index("cod_IBGE")
faltam = mun["Indicador faltante"].dropna().value_counts()
print(len(mun), "municípios;", mun["Nº faltantes"].eq(1).sum(), "com um faltante")
pd.crosstab(mun["Porte"], mun["Indicador faltante"].fillna("nenhum"))

581 municípios; 76 com um faltante


Indicador faltante,IAG0001,IES2003,IRS0001,nenhum
Porte,,,,
Grande porte,2,0,2,60
Pequeno e médio porte,6,3,63,445


## 1. Relação entre os indicadores

Correlações fracas indicam quanto os demais indicadores podem ajudar a prever o faltante.

In [2]:
mun[INDICADORES].corr().round(2)

,IAG0001,IES0001,IES2003,IRS0001,IRS3002
IAG0001,1.00,0.15,0.09,0.34,-0.04
IES0001,0.15,1.00,0.33,0.11,0.19
IES2003,0.09,0.33,1.00,0.13,-0.06
IRS0001,0.34,0.11,0.13,1.00,0.05
IRS3002,-0.04,0.19,-0.06,0.05,1.00


## 2. Estimador KNN dentro do porte

Os indicadores usados como preditores são padronizados (média 0, desvio 1) para que nenhum domine a distância. A distância é euclidiana, e o peso de cada vizinho é o inverso da distância. Um vizinho idêntico (distância zero) recebe o peso total.

In [3]:
def padroniza(X_ref, X):
    media, desvio = X_ref.mean(axis=0), X_ref.std(axis=0)
    return (X_ref - media) / desvio, (X - media) / desvio


def knn_porte(X_tr, y_tr, porte_tr, X_te, porte_te, k):
    # Estima y para X_te usando só doadores do mesmo porte; devolve as estimativas e os índices dos vizinhos
    estimativas = np.empty(len(X_te))
    vizinhos = [None] * len(X_te)
    for porte in np.unique(porte_te):
        doa = np.where(porte_tr == porte)[0]
        alvo = np.where(porte_te == porte)[0]
        Z_doa, Z_alvo = padroniza(X_tr[doa], X_te[alvo])
        nn = NearestNeighbors(n_neighbors=min(k, len(doa))).fit(Z_doa)
        dist, idx = nn.kneighbors(Z_alvo)
        for i, (d, j) in enumerate(zip(dist, idx)):
            peso = np.where(d == 0, 1.0, 0.0) if (d == 0).any() else 1 / d
            estimativas[alvo[i]] = np.average(y_tr[doa[j]], weights=peso)
            vizinhos[alvo[i]] = doa[j]
    return estimativas, vizinhos

## 3. Validação cruzada

Para cada indicador que falta em algum município (IAG0001, IES2003 e IRS0001), os doadores são os municípios selecionados em que o indicador e os quatro preditores são conhecidos. O erro é medido pelo erro absoluto médio (MAE) e pela raiz do erro quadrático médio (RMSE), em pontos percentuais.

In [4]:
def doadores(alvo):
    preditores = [c for c in INDICADORES if c != alvo]
    base = mun[mun[alvo].notna() & mun[preditores].notna().all(axis=1)]
    return base, preditores


def metodos(k_valores):
    def mediana(X_tr, y_tr, p_tr, X_te, p_te):
        return np.array([np.median(y_tr[p_tr == p]) for p in p_te])

    def regressao(X_tr, y_tr, p_tr, X_te, p_te):
        A_tr = np.c_[X_tr, p_tr == "Grande porte"]
        A_te = np.c_[X_te, p_te == "Grande porte"]
        return np.clip(LinearRegression().fit(A_tr, y_tr).predict(A_te), 0, 100)

    def floresta(X_tr, y_tr, p_tr, X_te, p_te):
        A_tr = np.c_[X_tr, p_tr == "Grande porte"]
        A_te = np.c_[X_te, p_te == "Grande porte"]
        modelo = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=SEMENTE, n_jobs=-1)
        return modelo.fit(A_tr, y_tr).predict(A_te)

    m = {"mediana do porte": mediana, "regressão linear": regressao, "random forest": floresta}
    for k in k_valores:
        m[f"KNN no porte, k = {k}"] = lambda *a, k=k: knn_porte(*a, k)[0]
    return m


resultados = []
for alvo in faltam.index:
    base, preditores = doadores(alvo)
    X, y, porte = base[preditores].to_numpy(float), base[alvo].to_numpy(float), base["Porte"].to_numpy()
    for nome, f in metodos(VALORES_K).items():
        erros = []
        for tr, te in RepeatedKFold(n_splits=10, n_repeats=5, random_state=SEMENTE).split(X):
            erros.append(f(X[tr], y[tr], porte[tr], X[te], porte[te]) - y[te])
        e = np.concatenate(erros)
        resultados.append({"indicador": alvo, "método": nome, "MAE": np.abs(e).mean(), "RMSE": np.sqrt((e ** 2).mean())})

cv = pd.DataFrame(resultados)
tabela = cv.pivot(index="método", columns="indicador", values=["MAE", "RMSE"]).round(2)
tabela[("MAE", "média")] = cv.pivot(index="método", columns="indicador", values="MAE").mean(axis=1).round(2)
tabela.sort_values(("MAE", "média"))

MAE                    RMSE                    MAE
indicador            IAG0001 IES2003 IRS0001 IAG0001 IES2003 IRS0001  média
método                                                                     
random forest          13.81   19.94   13.85   17.49   27.69   17.58  15.87
KNN no porte, k = 15   13.93   20.46   13.67   17.76   27.58   17.33  16.02
KNN no porte, k = 20   13.90   20.67   13.60   17.69   27.51   17.18  16.06
KNN no porte, k = 10   14.08   20.37   13.77   17.88   27.93   17.54  16.07
KNN no porte, k = 30   13.94   20.95   13.70   17.64   27.37   17.14  16.19
KNN no porte, k = 5    14.27   20.76   13.94   18.29   29.23   17.98  16.32
regressão linear       14.61   21.78   14.15   18.34   27.60   17.67  16.84
mediana do porte       15.53   21.74   14.49   19.40   31.69   18.78  17.26

## 4. Escolha de *k*

O *k* escolhido é o de menor MAE médio entre os três indicadores. A tabela acima também mostra o ganho em relação à mediana do porte, que era a regra prevista inicialmente.

In [5]:
mae_knn = cv[cv["método"].str.startswith("KNN")].groupby("método")["MAE"].mean()
K = int(mae_knn.idxmin().split("= ")[1])
mae_mediana = cv[cv["método"] == "mediana do porte"]["MAE"].mean()
print(f"k escolhido: {K} (MAE médio {mae_knn.min():.2f} p.p., contra {mae_mediana:.2f} p.p. da mediana do porte)")

k escolhido: 15 (MAE médio 16.02 p.p., contra 17.26 p.p. da mediana do porte)


## 5. Imputação

Cada valor faltante é estimado com os doadores do mesmo porte. A tabela mostra o valor imputado e os cinco vizinhos mais próximos que serviram de referência.

In [6]:
matriz = mun.copy()
matriz["Imputado"] = ""
matriz["Vizinhos (5 mais próximos)"] = ""

for alvo in faltam.index:
    base, preditores = doadores(alvo)
    receptores = mun[mun["Indicador faltante"] == alvo]
    est, viz = knn_porte(base[preditores].to_numpy(float), base[alvo].to_numpy(float), base["Porte"].to_numpy(),
                         receptores[preditores].to_numpy(float), receptores["Porte"].to_numpy(), K)
    matriz.loc[receptores.index, alvo] = est.round(2)
    matriz.loc[receptores.index, "Imputado"] = alvo
    matriz.loc[receptores.index, "Vizinhos (5 mais próximos)"] = [
        ", ".join(f"{base['Município'].iloc[j]} ({base['UF'].iloc[j]})" for j in v[:5]) for v in viz
    ]

assert matriz[INDICADORES].notna().all().all()
imputados = matriz[matriz["Imputado"] != ""]
imputados[["Município", "UF", "Porte", "Imputado"] + INDICADORES + ["Vizinhos (5 mais próximos)"]] \
    .sort_values(["Imputado", "UF", "Município"])

,Município,UF,Porte,Imputado,IAG0001,IES0001,IES2003,IRS0001,IRS3002,Vizinhos (5 mais próximos)
cod_IBGE,,,,,,,,,,
2704500,Maragogi,AL,Pequeno e médio porte,IAG0001,74.45,50.61,60.08,78.97,100.0,"Mata de São João (BA), Cravolândia (BA), São F..."
2706604,Paulo Jacinto,AL,Pequeno e médio porte,IAG0001,61.69,22.16,26.91,79.53,100.0,"Tacaratu (PE), Jacuípe (AL), Moreno (PE), Brej..."
2917508,Jacobina,BA,Pequeno e médio porte,IAG0001,71.86,41.16,36.85,59.65,0.0,"Camacan (BA), Ipirá (BA), Jaguaribara (CE), Bo..."
2929800,Saúde,BA,Pequeno e médio porte,IAG0001,75.93,1.87,0.54,65.33,0.0,"Riachão do Jacuípe (BA), Cascavel (CE), Amonta..."
2304202,Crato,CE,Grande porte,IAG0001,81.71,37.40,18.24,100.00,0.0,"Juazeiro do Norte (CE), Imperatriz (MA), Maran..."
...,...,...,...,...,...,...,...,...,...,...
2406106,Jucurutu,RN,Pequeno e médio porte,IRS0001,74.06,19.11,25.81,76.65,0.0,"Russas (CE), Jardim de Piranhas (RN), Bela Cru..."
2406809,Lajes Pintadas,RN,Pequeno e médio porte,IRS0001,76.35,27.62,38.26,79.73,0.0,"Pacatuba (CE), Jardim de Piranhas (RN), Jaguar..."
2410900,Riachuelo,RN,Pequeno e médio porte,IRS0001,74.64,56.69,77.39,87.13,0.0,"Várzea Nova (BA), Currais Novos (RN), Itamaraj..."


## 6. Conferência das distribuições

Os valores imputados devem ficar dentro da faixa observada nos doadores. Como o KNN calcula uma média, ele tende a puxar os valores para o centro da distribuição, e isso é esperado.

In [7]:
linhas = []
for alvo in faltam.index:
    base, _ = doadores(alvo)
    imp = matriz.loc[matriz["Imputado"] == alvo, alvo]
    for nome, s in [("doadores", base[alvo]), ("imputados", imp)]:
        linhas.append({"indicador": alvo, "grupo": nome, "n": len(s), "média": s.mean(), "mediana": s.median(),
                       "mín": s.min(), "máx": s.max()})
pd.DataFrame(linhas).set_index(["indicador", "grupo"]).round(2)

n  média  mediana    mín     máx
indicador grupo                                        
IRS0001   doadores   505  81.63    87.11  18.67  100.00
          imputados   65  80.37    81.01  62.03   93.26
IAG0001   doadores   505  69.58    70.66   0.00  100.00
          imputados    8  72.43    75.19  55.79   81.71
IES2003   doadores   505  24.74    12.65   0.00  100.00
          imputados    3  12.34    12.28  12.20   12.54

## 7. Exportação

A matriz de decisão traz os cinco critérios completos (o IRS3002 já adaptado) e a coluna `Imputado`, que indica qual indicador foi estimado.

In [8]:
colunas = ["Município", "UF", "Porte", "Categoria"] + INDICADORES + ["Imputado", "Vizinhos (5 mais próximos)"]
matriz[colunas].to_csv(SAIDA, encoding="utf-8-sig")
print(f"{len(matriz)} municípios, {len(imputados)} com valor imputado. Salvo em {SAIDA}")

581 municípios, 76 com valor imputado. Salvo em /home/user/Rankeamento-Municipios-Saneamento/resultados/matriz_decisao.csv
